# 01. Ingestão de Dados | Camada Bronze

Este notebook é responsável pela ingestão dos dados brutos utilizados no MVP.
Os dados analisados são provenientes da Organização Mundial da Saúde (OMS), disponibilizados por meio da plataforma Our World in Data, e contêm estimativas de prevalência de obesidade e excesso de peso na população adulta, segmentadas por sexo e país.
A camada Bronze tem como objetivo preservar os dados o mais próximo possível de sua estrutura original, sem aplicação de regras de negócio ou transformações analíticas.

## 1. Definição da origem dos dados

Os quatro arquivos CSV utilizados neste projeto foram armazenados em um Volume do Unity Catalog no Databricks.
Todos os arquivos estão localizados no mesmo diretório de origem.

In [0]:
base_path = "/Volumes/workspace/default/sisvan"

## 2. Leitura dos arquivos de origem

Foram utilizados quatro conjuntos de dados:
- obesidade em mulheres;
- excesso de peso em mulheres;
- obesidade em homens;
- excesso de peso em homens.

Cada arquivo é carregado individualmente com Apache Spark.
Nesta etapa, a inferência automática de esquema foi desabilitada para preservar os dados em sua estrutura original. A conversão dos tipos será realizada posteriormente na camada Silver.

In [0]:
df_obesity_female = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(f"{base_path}/share-of-females-defined-as-obese.csv")
)

df_overweight_female = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(f"{base_path}/share-of-females-defined-as-overweight.csv")
)

df_obesity_male = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(f"{base_path}/share-of-men-defined-as-obese.csv")
)

df_overweight_male = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(f"{base_path}/share-of-men-defined-as-overweight.csv")
)

## 3. Validação da quantidade de registros

Após a leitura dos arquivos, é realizada uma contagem de registros para verificar se os quatro conjuntos foram carregados corretamente.
Essa verificação também permite identificar possíveis diferenças inesperadas entre as fontes antes do início das transformações.

In [0]:
print("Obesidade - Mulheres:", df_obesity_female.count())
print("Excesso de peso - Mulheres:", df_overweight_female.count())
print("Obesidade - Homens:", df_obesity_male.count())
print("Excesso de peso - Homens:", df_overweight_male.count())

### Resultado da validação

Os quatro conjuntos de dados possuem 9.270 registros cada, totalizando 37.080 registros brutos.
A igualdade na quantidade de registros é compatível com a estrutura das fontes, uma vez que os conjuntos apresentam a mesma cobertura geográfica e temporal, diferenciando-se pelo sexo e pelo indicador nutricional analisado.

## 4. Inspeção das colunas

Os quatro arquivos compartilham os campos `Entity`, `Code` e `Year`.
Entretanto, a coluna responsável pelo valor do indicador apresenta nomes diferentes em cada arquivo, pois incorpora a descrição completa do indicador, sexo e faixa etária.
Essa diferença será tratada na camaa Silver através da padronização do esquema.

In [0]:
print("OBESIDADE - MULHERES")
print(df_obesity_female.columns)

print("\nEXCESSO DE PESO - MULHERES")
print(df_overweight_female.columns)

print("\nOBESIDADE - HOMENS")
print(df_obesity_male.columns)

print("\nEXCESSO DE PESO - HOMENS")
print(df_overweight_male.columns)

## 5. Inspeção visual dos dados

Antes da persistência da camada Bronze, é realizada uma inspeção visual de uma das fontes para confirmar a estrutura dos registros carregados.

In [0]:
display(df_obesity_female.limit(20))

## 6. Inspeção do esquema

Como esperado para a camada Bronze, os campos foram mantidos inicialmente como texto.
A conversão de `Year` para número inteiro e do indicador de prevalência para tipo numérico será realizada na camada Silver.

In [0]:
df_obesity_female.printSchema()

## 7. Persistência da camada Bronze

Após a validação da ingestão, cada conjunto de dados é persistido individualmente como uma tabela Delta.
As tabelas Bronze preservam a estrutura original de cada fonte e servem como ponto de partida confiável e reproduzível para as próximas etapas do pipeline.

### Adequação técnica do esquema

Os arquivos de origem possuem nomes de colunas contendo caracteres especiais não aceitos diretamente pelo formato Delta.
Para viabilizar a persistência das tabelas, os nomes dos campos foram padronizados para `entity`, `code`, `year` e `raw_value`.
Essa adequação modifica apenas os nomes das colunas e não altera os valores originais dos dados.

In [0]:
df_obesity_female = df_obesity_female.toDF(
    "entity",
    "code",
    "year",
    "raw_value"
)

df_overweight_female = df_overweight_female.toDF(
    "entity",
    "code",
    "year",
    "raw_value"
)

df_obesity_male = df_obesity_male.toDF(
    "entity",
    "code",
    "year",
    "raw_value"
)

df_overweight_male = df_overweight_male.toDF(
    "entity",
    "code",
    "year",
    "raw_value"
)

In [0]:
df_obesity_female.printSchema()

In [0]:
df_obesity_female.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_obesity_female")

df_overweight_female.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_overweight_female")

df_obesity_male.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_obesity_male")

df_overweight_male.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_overweight_male")

## 8. Verificação das tabelas persistidas

Por fim, é realizada uma consulta ao catálogo do Databricks para verificar se as quatro tabelas Bronze foram criadas com sucesso.

In [0]:
display(
    spark.sql("""
        SHOW TABLES IN workspace.default
    """)
)

## Conclusão

A etapa de ingestão da camada Bronze foi concluída.

Nesta etapa:
- os quatro arquivos CSV foram carregados a partir de um Volume do Unity Catalog;
- cada fonte foi processada separadamente;
- a quantidade de registros foi validada;
- a estrutura das colunas e o esquema foram inspecionados;
- os dados foram persistidos em tabelas Delta.
Na próxima etapa, os dados serão tratados na camada Silver, onde serão realizadas a padronização dos nomes das colunas, conversão dos tipos de dados, inclusão das dimensões de sexo e indicador e unificação dos quatro conjuntos.